<a href="https://colab.research.google.com/github/JxCorona/CIS3120/blob/main/JC_CIS3120_Class13_SQL_Notebook_11_Mar_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS 3120 — Programming for Analytics
## Class 13: Introduction to SQL and Sub-Queries
**Spring 2026**

---

This notebook accompanies the Class 13 lesson plan. Work through each section in order. Explanatory text and examples are provided before every exercise. Where you see a `# TODO` comment, that is where you write your own code.

**Two reminders before you begin:**
1. SQL statements end with a semicolon `;`
2. In Python, SQL is passed as a *string* — either in single quotes, double quotes, or triple quotes for multi-line statements


---
### Resources

| Resource | Link |
|---|---|
| Charles Severance — Databases (py4e) | https://www.py4e.com/lessons/database |
| practical_db — SQL Basics (Chapter 1) | https://runestone.academy/ns/books/published/practical_db/PART1_SQL/01-sql-basics/sql-basics.html |
| W3Resource — SQL Subqueries | https://www.w3resource.com/sql/subqueries/understanding-sql-subqueries.php |
| DataQuest — SQL Subqueries for Beginners | https://www.dataquest.io/blog/sql-subqueries-for-beginners/ |


---
## Part 1 — What Is SQL, and Why Does It Matter?

**SQL** (Structured Query Language) is the standard language for communicating with *relational databases* — organized collections of data stored in tables of rows and columns.

### Why not just use Pandas?

Pandas operates on data that has been loaded *into memory* on your machine. That works fine for small datasets, but enterprise databases often contain millions or billions of rows — far too large to load at once. SQL runs *on the database server*, which means filtering and aggregation happen before any data travels to your program.

In practice, a business analyst's workflow often looks like:

```
SQL query (server-side) → small, filtered result → load into Pandas for visualization
```

### Python's sqlite3 module

`sqlite3` is part of Python's standard library — no installation required. It allows Python to create, read, and modify SQLite database files (`.db` files) directly on disk.

The four-step pattern you will use in every database interaction:

| Step | Code | Purpose |
|---|---|---|
| 1 | `conn = sqlite3.connect('mydb.db')` | Open (or create) a database file |
| 2 | `conn.execute(...)` or `cursor.execute(...)` | Run a SQL statement |
| 3 | `conn.commit()` | Save any changes permanently |
| 4 | `conn.close()` | Release the file |

> **Important:** Steps 3 and 4 are easy to forget. Skipping `commit()` means your `INSERT` or `CREATE` statements are lost when the connection closes. Always close connections when you are done.


---
### Housekeeping — Import Libraries


In [ ]:
import sqlite3
import os
import urllib.request  # used to download files from GitHub


---
## Part 2 — Core SQL Syntax: CREATE, INSERT, SELECT

We will build a small database about soccer team performance to learn the basics before moving on to the exercises.

### Step 1 — Connect to (or create) a database file


In [ ]:
# sqlite3.connect() opens an existing file, or creates a new one if it does not exist.

try:
    conn = sqlite3.connect('soccer.db')
    print("Connected to soccer.db successfully.")
except Exception as e:
    print(f"Connection failed: {e}")


Connected to soccer.db successfully.


### Step 2 — CREATE TABLE

The `CREATE TABLE` statement defines the structure of a table — its column names and data types.

`IF NOT EXISTS` prevents an error if you run this cell more than once.

**Common SQLite data types:**

| Type | Use for |
|---|---|
| `TEXT` | Names, codes, labels |
| `INTEGER` | Whole numbers (counts, IDs, years) |
| `REAL` | Decimal numbers (prices, rates) |


In [ ]:
try:
    conn.execute('''
        CREATE TABLE IF NOT EXISTS team_data (
            team         TEXT,
            country      TEXT,
            season       INTEGER,
            total_goals  INTEGER
        );
    ''')
    conn.commit()
    print("Table created successfully.")
except Exception as e:
    print(f"Error: {e}")


Table created successfully.


### Step 3 — INSERT rows

`INSERT INTO` adds one row at a time. The values must match the column order defined in `CREATE TABLE`.


In [ ]:
try:
    conn.execute("INSERT INTO team_data VALUES ('Real Madrid', 'Spain', 2019, 53);")
    conn.execute("INSERT INTO team_data VALUES ('Barcelona',   'Spain', 2019, 47);")
    conn.execute("INSERT INTO team_data VALUES ('Arsenal',     'UK',    2019, 52);")
    conn.execute("INSERT INTO team_data VALUES ('Real Madrid', 'Spain', 2018, 49);")
    conn.execute("INSERT INTO team_data VALUES ('Barcelona',   'Spain', 2018, 45);")
    conn.execute("INSERT INTO team_data VALUES ('Arsenal',     'UK',    2018, 50);")
    conn.commit()
    print("All rows inserted.")
except Exception as e:
    print(f"Error: {e}")


All rows inserted.


### Step 4 — SELECT: retrieve data

`SELECT` is the most common SQL statement. The basic pattern is:

```sql
SELECT column1, column2
FROM   table_name
WHERE  condition
ORDER BY column1 ASC|DESC;
```

- Use `*` in place of column names to return all columns.
- `WHERE` filters rows *before* any results are returned.
- `ORDER BY` sorts the result; default is ascending (`ASC`).


In [ ]:
# SELECT all columns from team_data
cursor = conn.execute('''
    SELECT *
    FROM   team_data;
''')

print("All rows in team_data:")
for row in cursor:
    print(row)


All rows in team_data:
('Real Madrid', 'Spain', 2019, 53)
('Barcelona', 'Spain', 2019, 47)
('Arsenal', 'UK', 2019, 52)
('Real Madrid', 'Spain', 2018, 49)
('Barcelona', 'Spain', 2018, 45)
('Arsenal', 'UK', 2018, 50)


In [ ]:
# SELECT with WHERE: only rows from Spain
cursor = conn.execute('''
    SELECT team, season, total_goals
    FROM   team_data
    WHERE  country = 'Spain';
''')

print("Spanish teams only:")
for row in cursor:
    print(row)


Spanish teams only:
('Real Madrid', 2019, 53)
('Barcelona', 2019, 47)
('Real Madrid', 2018, 49)
('Barcelona', 2018, 45)


In [ ]:
# SELECT with ORDER BY: highest scorers first
cursor = conn.execute('''
    SELECT team, season, total_goals
    FROM   team_data
    ORDER BY total_goals DESC;
''')

print("All teams, highest goals first:")
for row in cursor:
    print(row)


All teams, highest goals first:
('Real Madrid', 2019, 53)
('Arsenal', 2019, 52)
('Arsenal', 2018, 50)
('Real Madrid', 2018, 49)
('Barcelona', 2019, 47)
('Barcelona', 2018, 45)


---
## Part 3 — Aggregate Functions and GROUP BY

Aggregate functions *collapse* multiple rows into a single summary value.

| Function | Returns |
|---|---|
| `COUNT(*)` | Number of rows |
| `COUNT(col)` | Number of non-NULL values in col |
| `SUM(col)` | Total of all values |
| `AVG(col)` | Arithmetic mean |
| `MIN(col)` | Smallest value |
| `MAX(col)` | Largest value |

`GROUP BY` controls the *granularity* of aggregation — one output row is produced per distinct value of the grouped column.

```sql
SELECT   team, AVG(total_goals) AS avg_goals
FROM     team_data
GROUP BY team;
```

The `AS` keyword assigns an *alias* — a friendlier name for the computed column.


In [ ]:
# Average goals per team across all seasons
cursor = conn.execute('''
    SELECT   team,
             AVG(total_goals) AS avg_goals
    FROM     team_data
    GROUP BY team;
''')

print("Average goals by team:")
for row in cursor:
    print(row)


Average goals by team:
('Arsenal', 51.0)
('Barcelona', 46.0)
('Real Madrid', 51.0)


### HAVING vs WHERE — a critical distinction

`WHERE` filters *individual rows* before grouping takes place.
`HAVING` filters *groups* after aggregation is complete.

This means you **cannot** use `WHERE` to filter on an aggregate alias like `avg_goals`.
The cell below demonstrates the error intentionally — read the error message carefully.


In [ ]:
# ── This query will produce an error — that is expected ──
# WHERE cannot reference avg_goals because it does not exist yet
# when WHERE is evaluated.

try:
    cursor = conn.execute('''
        SELECT   team,
                 AVG(total_goals) AS avg_goals
        FROM     team_data
        GROUP BY team
        where    avg_goals > 50;
    ''')
    for row in cursor:
        print(row)
except Exception as e:
    print(f"Error (expected): {e}")


Error (expected): near "where": syntax error


In [ ]:
# ── The correct approach: use HAVING ──
# HAVING is evaluated AFTER GROUP BY, so it can reference aggregate aliases.

cursor = conn.execute('''
    SELECT   team,
             AVG(total_goals) AS avg_goals
    FROM     team_data
    GROUP BY team
    HAVING   avg_goals > 50;
''')

print("Teams with average goals > 50:")
for row in cursor:
    print(row)


Teams with average goals > 50:
('Arsenal', 51.0)
('Real Madrid', 51.0)


---
## Part 4 — Sub-Queries

A **sub-query** is a complete SQL query nested inside another query. The inner query executes first; its result is then used by the outer query.

### The inline-view pattern

The most common pattern for business analytics places the sub-query in the `FROM` clause, where it acts as a *temporary named table*:

```sql
SELECT outer_col1, outer_col2
FROM (
    -- inner query produces a temporary table
    SELECT col1,
           AGG(col2) AS alias
    FROM   source_table
    GROUP BY col1
) AS alias_for_temp_table
WHERE outer_col2 > threshold;
```

The outer `WHERE` clause *can* reference `alias` because, from the outer query's perspective, the sub-query result is just another table.

### Why use a sub-query instead of HAVING?

`HAVING` covers the most common case. Sub-queries are needed when you:
- Must *join* an aggregated result to another table
- Need to apply additional transformations to the aggregated result
- Are working in a context where the database system does not support `HAVING` on aliases

The pattern is worth learning because it generalises far beyond the `HAVING` use case.


#### The Role of tp in This SQL Statement
`tp` is a table alias assigned to the subquery (the inner SELECT statement).  

<u>Why It Is Required</u>  
In SQL, when a subquery appears in the `FROM` clause, the database engine materializes it as a temporary, unnamed result set. Most SQL engines — including SQLite, which is what sqlite3 uses — require that this temporary result set be given a name so that the outer query has something to reference.  
  
<i>Without the `AS tp`, SQLite would raise a syntax error.</i>  

In [ ]:
# Sub-query: filter on an average computed in the inner query

cursor = conn.execute('''
    SELECT team_name, avg_goals
    FROM (
        -- ── INNER QUERY: aggregate first ──────────────────────────
        SELECT   team             AS team_name,
                 AVG(total_goals) AS avg_goals
        FROM     team_data
        GROUP BY team
        -- ── end of inner query ────────────────────────────────────
    ) AS tp
    WHERE avg_goals > 50;
''')

print("Teams with avg goals > 50 (via sub-query):")
for row in cursor:
    print(row)


Teams with avg goals > 50 (via sub-query):
('Arsenal', 51.0)
('Real Madrid', 51.0)


If there were any ambiguity, or if you needed to qualify the columns explicitly, you would write:  
```
SELECT tp.team_name, tp.avg_goals
FROM ( ... ) AS tp
WHERE tp.avg_goals > 50;
```

In [ ]:
# Sub-query: filter on an average computed in the inner query

cursor = conn.execute('''
    SELECT tp.team_name, tp.avg_goals
    FROM (
        -- ── INNER QUERY: aggregate first ──────────────────────────
        SELECT   team             AS team_name,
                 AVG(total_goals) AS avg_goals
        FROM     team_data
        GROUP BY team
        -- ── end of inner query ────────────────────────────────────
    ) AS tp
    WHERE tp.avg_goals > 50;
''')

print("Teams with avg goals > 50 (via sub-query) using dot notation on `tp`:")
for row in cursor:
    print(row)


Teams with avg goals > 50 (via sub-query) using dot notation on `tp`:
('Arsenal', 51.0)
('Real Madrid', 51.0)


In [ ]:
# Close the soccer.db connection — we are done with the demonstration data.
conn.close()
print("soccer.db connection closed.")


soccer.db connection closed.


---
## Part 5 — Loading a Database from an External SQL File

In professional settings, SQL is rarely typed directly into Python strings. Instead, SQL statements are stored in `.sql` files that can be version-controlled, reviewed, and reused across projects.

Python's `cursor.executescript()` method reads and executes an entire SQL file in one call.

### Downloading a file from GitHub

When a `.sql` file is stored in a GitHub repository, you can download it using its **raw** URL — the URL that serves the plain text content of the file. Raw URLs follow the pattern:

```
https://raw.githubusercontent.com/<username>/<repo>/<branch>/<path/to/file.sql>
```

The cell below uses `urllib.request.urlretrieve()` to download the file to the Colab runtime before executing it.

### A note on SQL dialects

The SQL standard is implemented slightly differently by each database system. MySQL uses `CREATE DATABASE` and `USE <database>` to select a working database. **SQLite does not support these statements** — in SQLite, the "database" is simply the `.db` file you connect to. When working with SQL files written for MySQL, these lines must be removed before passing the script to `executescript()`.

The cell below handles this automatically.


In [ ]:
# ── Download the SQL script from GitHub ──────────────────────────────────────
#
# Replace GITHUB_RAW_URL with the actual raw GitHub URL for your SQL file.
# Raw URL format:
#   https://raw.githubusercontent.com/<username>/<repo>/refs/heads/main/<file.sql>

GITHUB_RAW_URL = "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/SQLite/refs/heads/main/SQL_create_badPINs.sql"
LOCAL_SQL_PATH = "/content/SQL_create_badPINs.sql"

try:
    urllib.request.urlretrieve(GITHUB_RAW_URL, LOCAL_SQL_PATH)
    print(f"Downloaded SQL file to: {LOCAL_SQL_PATH}")
except Exception as e:
    print(f"Download failed: {e}")
    print("Writing a local SQLite-compatible fallback...")
    with open(LOCAL_SQL_PATH, 'w') as f:
        f.write(
            'CREATE TABLE IF NOT EXISTS COMMON_PINS (\n'
            '    "NUMBER"    INTEGER NOT NULL UNIQUE,\n'
            '    "PIN_VALUE" INTEGER NOT NULL UNIQUE,\n'
            '    PRIMARY KEY("NUMBER" AUTOINCREMENT)\n'
            ');\n'
        )
    print("Fallback file written.")


Downloaded SQL file to: /content/SQL_create_badPINs.sql


In [ ]:
# ── Inspect the downloaded SQL file before running it ────────────────────────
# Read the file first so you can see exactly what statements it contains.

with open(LOCAL_SQL_PATH, 'r') as f:
    raw_sql = f.read()

print("Raw contents of the SQL file:")
print("-" * 50)
print(raw_sql)


Raw contents of the SQL file:
--------------------------------------------------
-- remove the comment on CREATE statement in line 3 if using MySQL
-- comment the next line if not using MySQL
-- CREATE DATABASE IF NOT EXISTS bad_pins;

-- remove the comment on the USE statement in line 7 if using MySQL
-- comment he next line if not using MySQL
-- USE bad_pins;

CREATE TABLE "COMMON_PINS" (
	"NUMBER"	INTEGER NOT NULL UNIQUE,
	"PIN_VALUE"	INTEGER NOT NULL UNIQUE,
	PRIMARY KEY("NUMBER" AUTOINCREMENT)
);







### Preparing the script for SQLite

The SQL file above was written with MySQL in mind. Before passing it to SQLite's `executescript()`, two MySQL-only statements can be removed, though they should be commented out and not a problem:

| Statement | Why it fails in SQLite with MySQL statements |
|---|---|
| `CREATE DATABASE IF NOT EXISTS bad_pins;` | SQLite has no `CREATE DATABASE` command — the database *is* the `.db` file. |
| `USE bad_pins;` | SQLite has no `USE` command — the active database is determined by which file you connected to. |

The cell below strips those lines, prints the cleaned script so you can verify it, and then executes it.


In [ ]:
# ── Strip MySQL-incompatible statements and execute ───────────────────────────

# Lines to remove — these are MySQL-only and cause SQLite to abort silently
MYSQL_ONLY = [
    'CREATE DATABASE IF NOT EXISTS bad_pins;',
    'USE bad_pins;',
]

with open(LOCAL_SQL_PATH, 'r') as f:
    raw_lines = f.readlines()

# Keep only lines that are not MySQL-specific (strip inline comments and whitespace for comparison)
clean_lines = [
    line for line in raw_lines
    if line.strip().rstrip(';') not in [s.rstrip(';') for s in MYSQL_ONLY]
    and not line.strip().startswith('-- remove the comment')
]
clean_sql = ''.join(clean_lines)

print("Cleaned SQL (MySQL-only lines removed):")
print("-" * 50)
print(clean_sql)
print("-" * 50)

# Connect and execute
conn_pins = sqlite3.connect('bad_pins.db')
cursor_pins = conn_pins.cursor()

try:
    cursor_pins.executescript(clean_sql)
    conn_pins.commit()
    print("\nScript executed successfully.")

    # Confirm the table exists and show its structure
    cursor_pins.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor_pins.fetchall()
    print(f"Tables in bad_pins.db: {tables}")

except sqlite3.Error as e:
    print(f"SQLite error: {e}")
    conn_pins.rollback()
except Exception as e:
    print(f"Unexpected error: {e}")
finally:
    conn_pins.close()
    print("bad_pins.db connection closed.")


Cleaned SQL (MySQL-only lines removed):
--------------------------------------------------
-- comment the next line if not using MySQL
-- CREATE DATABASE IF NOT EXISTS bad_pins;

-- comment he next line if not using MySQL
-- USE bad_pins;

CREATE TABLE "COMMON_PINS" (
	"NUMBER"	INTEGER NOT NULL UNIQUE,
	"PIN_VALUE"	INTEGER NOT NULL UNIQUE,
	PRIMARY KEY("NUMBER" AUTOINCREMENT)
);





--------------------------------------------------
SQLite error: table "COMMON_PINS" already exists
bad_pins.db connection closed.


In [ ]:
# ── Insert sample bad PINs ────────────────────────────────────────────────────
# The SQL file creates the COMMON_PINS table structure only — it contains no data.
# The cells below insert a set of well-known bad PINs so Exercise 1 has rows to query.
#
# Note: PIN_VALUE is stored as INTEGER. Leading zeros are not preserved
# (e.g., 0000 is stored as 0). This is a property of the INTEGER data type.

bad_pins_data = [
    (1234,),
    (1111,),
    (0,),       # represents '0000'
    (1212,),
    (7777,),
    (1004,),
    (2000,),
    (4444,),
]

conn_pins = sqlite3.connect('bad_pins.db')
cursor_pins = conn_pins.cursor()

try:
    cursor_pins.executemany(
        "INSERT OR IGNORE INTO COMMON_PINS (PIN_VALUE) VALUES (?);",
        bad_pins_data
    )
    conn_pins.commit()
    print(f"Inserted {cursor_pins.rowcount} rows (duplicates skipped).")

    # Verify
    cursor_pins.execute("SELECT * FROM COMMON_PINS ORDER BY NUMBER;")
    rows = cursor_pins.fetchall()
    print(f"\nContents of COMMON_PINS ({len(rows)} rows):")
    print(f"  {'NUMBER':>8}  {'PIN_VALUE':>10}")
    print(f"  {'-'*8}  {'-'*10}")
    for row in rows:
        print(f"  {row[0]:>8}  {row[1]:>10}")

except sqlite3.Error as e:
    print(f"SQLite error: {e}")
    conn_pins.rollback()
finally:
    conn_pins.close()
    print("\nbad_pins.db connection closed.")


Inserted 0 rows (duplicates skipped).

Contents of COMMON_PINS (8 rows):
    NUMBER   PIN_VALUE
  --------  ----------
         1        1234
         2        1111
         3           0
         4        1212
         5        7777
         6        1004
         7        2000
         8        4444

bad_pins.db connection closed.


---
## ✏️ Exercise 1 — Query the COMMON_PINS Database

**Scenario:** Security researchers have compiled a list of the most commonly used ATM PIN codes — predictable PINs that make accounts easy to compromise. The SQL script you loaded in Part 5 created a table named `COMMON_PINS` in `bad_pins.db`, and sample data has been inserted. Your task is to query that table to answer three straightforward questions.

**Table structure:**

| Column | Type | Description |
|---|---|---|
| `NUMBER` | INTEGER | Auto-incrementing primary key |
| `PIN_VALUE` | INTEGER | The numeric PIN code |

**Before you begin:**
- Make sure all cells in Part 5 have been run (download → strip → execute → insert data).
- Re-open the database connection in the first cell below.
- Complete every `# TODO` comment.

---


In [ ]:
# ── Re-open the connection to bad_pins.db ────────────────────────────────────

conn_ex1 = sqlite3.connect('bad_pins.db')
print("Connected to bad_pins.db")


Connected to bad_pins.db


### Exercise 1 — Step 1

**Confirm the data loaded correctly.**

Run the cell below as-is. You should see 8 rows, with `NUMBER` values 1 through 8 and the corresponding `PIN_VALUE` integers.


In [ ]:
# Step 1: Confirm all rows — no changes needed here

cursor = conn_ex1.execute("SELECT * FROM COMMON_PINS ORDER BY NUMBER;")
print("All rows in COMMON_PINS:")
for row in cursor:
    print(row)


All rows in COMMON_PINS:
(1, 1234)
(2, 1111)
(3, 0)
(4, 1212)
(5, 7777)
(6, 1004)
(7, 2000)
(8, 4444)


### Exercise 1 — Step 2

**Retrieve only the `PIN_VALUE` column** (not the `NUMBER`).

Write a `SELECT` statement that returns just `PIN_VALUE` from `COMMON_PINS`, ordered from smallest to largest.


In [ ]:
# Step 2: SELECT only PIN_VALUE, ordered ascending

# TODO: Write a SELECT statement that retrieves only the PIN_VALUE column,
#       ordered from smallest to largest (ORDER BY PIN_VALUE ASC)
cursor = conn_ex1.execute(''' Select pin_value from common_pins order by pin_value asc ''')

print("PIN values only (ascending):")
for row in cursor:
    print(row)


PIN values only (ascending):
(0,)
(1004,)
(1111,)
(1212,)
(1234,)
(2000,)
(4444,)
(7777,)


### Exercise 1 — Step 3

**Count the total number of rows** in the table using `COUNT(*)`.

Your query should return a single number: `8`.


In [ ]:
# Step 3: COUNT all rows

# TODO: Write a SELECT statement using COUNT(*) to count all rows in COMMON_PINS
cursor = conn_ex1.execute(''' SELECT count(*) from COMMON_PINS as total_pins ''')

print("Total rows in COMMON_PINS:")
for row in cursor:
    print(row)


Total rows in COMMON_PINS:
(8,)


### Exercise 1 — Step 4

**Add a new bad PIN to the table.**

Insert one additional predictable PIN of your choice using `INSERT INTO COMMON_PINS (PIN_VALUE) VALUES (...)`, commit the change, then run `COUNT(*)` again to confirm the total is now `9`.

> **Hint:** You only need to supply `PIN_VALUE` — the `NUMBER` column fills itself automatically because it is defined as `AUTOINCREMENT`.


In [ ]:
# Step 4a: INSERT a new bad PIN
# TODO: Write an INSERT statement for a PIN_VALUE of your choice.
#       Example bad PINs not yet in the table: 9999, 6969, 1122, 1313

conn_ex1.execute(''' INSERT into COMMON_PINS ("pin_value") VALUES ("1996") ''')
conn_ex1.commit()
print("New PIN inserted.")


New PIN inserted.


In [ ]:
# Step 4b: COUNT again to confirm the new total (should be 9)

# TODO: Write a COUNT(*) query and print the result
cursor = conn_ex1.execute(''' SELECT count(*) from COMMON_PINS as total_pins ''')

print("Total rows after INSERT:")
for row in cursor:
    print(row)


Total rows after INSERT:
(9,)


In [ ]:
# Close the connection when finished with Exercise 1
conn_ex1.close()
print("bad_pins.db connection closed.")


bad_pins.db connection closed.


---
> **Check your work — Exercise 1**
>
> | Step | Expected result |
> |---|---|
> | Step 1 | 8 rows, `NUMBER` 1–8, `PIN_VALUE` matching the inserted data |
> | Step 2 | 8 tuples containing only `PIN_VALUE`, in ascending numeric order |
> | Step 3 | `(8,)` |
> | Step 4 | `(9,)` |
>
> **Design note:** `PIN_VALUE` is stored as `INTEGER`, so the PIN `0000` is stored as `0`. If preserving leading zeros matters (e.g., for display), the column type would need to be `TEXT`. This is a common trade-off in database design between storage type and display requirements.


---
## ✏️ Exercise 2 — Aggregate Filtering with HAVING

**Scenario:** You are reviewing the soccer team data from the lecture demonstration. Your manager wants to know which teams scored the most goals *in total across all recorded seasons*, and specifically wants to see only the teams that reached a combined total above 95 goals.

You will write this query in three stages, building up from a simple aggregation to a filtered, sorted result.

**Before you begin:** Re-open `soccer.db` in the cell below. The `team_data` table was created and populated in Part 2 of this notebook.

---


In [ ]:
# ── Re-open the connection to soccer.db ──────────────────────────────────────

conn_ex2 = sqlite3.connect('soccer.db')
print("Connected to soccer.db")


Connected to soccer.db


### Exercise 2 — Step 1

**Refresh your memory: display all rows** in `team_data` so you can verify your results in the steps that follow.


In [ ]:
# Step 1: Display all rows in team_data — no changes needed here

cursor = conn_ex2.execute('''
    SELECT *
    FROM   team_data
    ORDER BY team, season;
''')

print("All rows in team_data:")
for row in cursor:
    print(row)


All rows in team_data:
('Arsenal', 'UK', 2018, 50)
('Arsenal', 'UK', 2019, 52)
('Barcelona', 'Spain', 2018, 45)
('Barcelona', 'Spain', 2019, 47)
('Real Madrid', 'Spain', 2018, 49)
('Real Madrid', 'Spain', 2019, 53)


### Exercise 2 — Step 2

**Total goals per team across all seasons.**

Write a query that:
- Returns `team` and the sum of `total_goals` for each team across both seasons
- Aliases the sum as `total_goals_all_seasons`
- Groups by `team`

> Hint: Use `SUM(total_goals)` and `GROUP BY team`.
>
> Expected result (order may vary):
> - Arsenal: 102
> - Barcelona: 92
> - Real Madrid: 102


In [ ]:
# Step 2: SUM of goals per team

# TODO: Write a SELECT query using SUM(total_goals) AS total_goals_all_seasons
#       grouped by team
cursor = conn_ex2.execute(''' SELECT team, SUM(total_goals) as total_goals_all_seasons from team_data group by team ''')

print("Total goals by team:")
for row in cursor:
    print(row)


Total goals by team:
('Arsenal', 102)
('Barcelona', 92)
('Real Madrid', 102)


### Exercise 2 — Step 3

**Filter to teams with total goals above 95 using HAVING.**

Copy your query from Step 2 and add a `HAVING` clause that keeps only teams whose `total_goals_all_seasons` exceeds 95.

> Expected result: Arsenal (102) and Real Madrid (102). Barcelona (92) should not appear.


In [ ]:
# Step 3: Add HAVING total_goals_all_seasons > 95

# TODO: Copy your Step 2 query and add a HAVING clause
cursor = conn_ex2.execute(''' SELECT team, SUM(total_goals) as total_goals_all_seasons
from team_data group by team

having total_goals_all_seasons > 95  ''')

print("Teams with total goals > 95:")
for row in cursor:
    print(row)


Teams with total goals > 95:
('Arsenal', 102)
('Real Madrid', 102)


### Exercise 2 — Step 4

**Sort the results from highest to lowest total.**

Extend your query from Step 3 with an `ORDER BY` clause that sorts `total_goals_all_seasons` in descending order.


In [ ]:
# Step 4: Add ORDER BY total_goals_all_seasons DESC

# TODO: Copy your Step 3 query and add ORDER BY
cursor = conn_ex2.execute(''' SELECT team, SUM(total_goals) as total_goals_all_seasons
from team_data group by team
having total_goals_all_seasons > 95

order by total_goals_all_seasons desc  ''')

print("Teams with total goals > 95:")

print("Filtered and sorted results:")
for row in cursor:
    print(row)


Teams with total goals > 95:
Filtered and sorted results:
('Real Madrid', 102)
('Arsenal', 102)


In [ ]:
# Close the connection when finished with Exercise 2
conn_ex2.close()
print("soccer.db connection closed.")


soccer.db connection closed.


---
> **Check your work — Exercise 2**
>
> | Step | Expected result |
> |---|---|
> | Step 2 | Three rows: Arsenal 102, Barcelona 92, Real Madrid 102 (any order) |
> | Step 3 | Two rows: Arsenal 102, Real Madrid 102 |
> | Step 4 | Same two rows, sorted 102, 102 (both teams tied, order between them may vary) |

---

> **Connecting to the next topic:** Your `HAVING` solution works perfectly here. But consider: what if you needed to *join* this aggregated result to a separate table listing each team's home city? `HAVING` alone cannot do that — you would need to place the aggregation inside a sub-query in the `FROM` clause. That is exactly the pattern demonstrated in **Part 4** of this notebook.


---
## Summary

In this notebook you practised:

| Concept | Statement(s) used |
|---|---|
| Creating a database and table | `sqlite3.connect()`, `CREATE TABLE IF NOT EXISTS` |
| Adding data | `INSERT INTO ... VALUES` |
| Retrieving data | `SELECT`, `WHERE`, `ORDER BY` |
| Summarising data | `COUNT(*)`, `SUM()`, `AVG()`, `MIN()`, `MAX()`, `GROUP BY` |
| Filtering aggregated results | `HAVING` (not `WHERE`) |
| Nesting queries | Sub-query in `FROM` clause with alias |
| Loading SQL from a file | `cursor.executescript()` |
| Fetching a file from GitHub | `urllib.request.urlretrieve()` |

---

### Before next class

- Push this completed notebook to your course GitHub repository.
- Read **practical_db Chapter 1** in full and complete the embedded interactive exercises:
  https://runestone.academy/ns/books/published/practical_db/PART1_SQL/01-sql-basics/sql-basics.html
- Next class: **multi-table databases** — foreign keys, JOIN operations, and combining the data you worked with today.
